# Predicting UK Renewable Energy Planning Outcomes

As a local to the Park Royal and Old Oak Common area, my initial interest in this project was sparked by observing the rapid urban regeneration happening on my own doorstep. A prime example of this is the local district heating network, which was intended to capture waste heat from a newly built data centre to sustainably warm nearby homes.

Ambitious projects like this highlight how local planning authorities and development corporations (such as the OPDC) are increasingly prioritising green infrastructure, circular-economy hubs, and sustainable regeneration to meet the UK’s net-zero targets. However, deploying renewable energy projects, whether district heating, solar, wind, or battery storage, faces a significant bottleneck: the statutory planning process.

Navigating fragmented land ownership, community concerns, and stringent local regulations makes securing planning permission complex and expensive. A refused application represents a massive loss of time, capital, and potential 'green-collar' job creation. Understanding the hidden patterns behind which projects get approved and which get refused is critical for developers, local councils, and policymakers looking to de-risk investments and accelerate sustainable development.


Project Objectives

This project aims to reverse-engineer the UK's renewable energy planning decisions using Machine Learning. By analysing historical application data, including geographic regions, technology types, installed capacities, and NLP embeddings of site text. This notebook develops a predictive pipeline to classify whether a proposed project will be Approved or Refused.

Specifically, this project seeks to:

- Identify High-Risk Applications: Build a model capable of flagging likely 'Refusals' before millions of pounds are spent on development.
- Solve Severe Data Imbalance: Tackle the inherent 89:11 class imbalance in planning data (where approvals heavily outweigh refusals) by engineering custom loss functions and decision thresholds.
- Establish a Scientific Benchmark: Rigorously compare the performance of complex Deep Learning architectures (PyTorch Neural Networks) against highly interpretable, traditional baselines (Logistic Regression) to determine the most efficient approach for this specific tabular dataset.


Key Technical Highlights

- End-to-End Pipeline: Data cleaning, robust leakage prevention, and advanced imputation strategies.
- Deep Learning (PyTorch): Implementation of a custom Neural Network, bypassing standard activation functions in favour of BCEWithLogitsLoss for mathematical stability.
- Advanced Imbalance Handling: Utilisation of dynamic class weighting (optimising for the minority 'Refused' class) combined with post-training probability threshold tuning to maximise the Macro F1 Score.
- NLP Integration: Incorporating text embeddings to capture the semantic nuances of operator and site names.

## Part 1: Data Readiness

In [1]:
# pip install -r requirements.txt

In [2]:
from local_config import REPD_DATASET
from helpers import upload_dataframe_from_csv, find_file_encoding, ml_ready_check

In [3]:
import os
os.environ["LOKY_MAX_CPU_COUNT"] = "16"

In [4]:
import numpy as np
import pandas as pd
import seaborn as sns; sns.set_theme() 
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer, normalize
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score, roc_auc_score, average_precision_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight
import copy

import charset_normalizer
import tempfile, joblib
from typing import Tuple
import warnings; warnings.filterwarnings("ignore")

from xgboost import XGBClassifier

In [5]:
RANDOM_STATE = 9999

The file encoding is identified before loading the dataset.

In [6]:
find_file_encoding(REPD_DATASET)

{'encoding': 'cp775', 'language': 'English', 'confidence': 1.0}
Consider trying a more common encoding like 'utf-8', 'latin1', or 'windows-1252' if there is an erroropening this file.


The dataset uses cp775 encoding. Now we can correctly open this file and check its contents.

In [7]:
df = pd.read_csv(REPD_DATASET, encoding="cp775")

df.head(5)

,Old Ref ID,Ref ID,Record Last Updated (dd/mm/yyyy),Operator (or Applicant),Site Name,Technology Type,Storage Type,Storage Co-location REPD Ref ID,Installed Capacity (MWelec),Share Community Scheme,...,Appeal Granted,Planning Permission Granted,Secretary of State - Intervened,Secretary of State - Refusal,Secretary of State - Granted,Planning Permission Expired,Under Construction,Operational,Heat Network Ref,Solar Site Area (sqm)
0,1,10726459,07/07/2009,RWE npower,Aberthaw Power Station Biomass,Biomass (co-firing),NaN,NaN,35.00,NaN,...,NaN,03/09/2004,NaN,NaN,NaN,NaN,01/05/2006,01/05/2007,NaN,NaN
1,2,NaN,20/11/2017,Orsted (formerly Dong Energy) / Peel Energy,Hunterston - cofiring,Biomass (co-firing),NaN,NaN,170.00,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,12019680,20/12/2019,Scottish and Southern Energy (SSE),Ferrybridge Multifuel 2 (FM2),EfW Incineration,NaN,NaN,70.00,NaN,...,NaN,28/10/2015,NaN,NaN,NaN,28/10/2020,01/09/2016,20/12/2019,NaN,NaN
3,4,11877116,18/12/2003,Energy Power Resources,Thetford Biomass Power Station,Biomass (dedicated),NaN,NaN,38.50,NaN,...,NaN,05/05/1995,NaN,NaN,NaN,NaN,NaN,02/10/1998,NaN,NaN
4,5,NaN,29/09/2005,Agrigen,Nunn Mills Road Biomass Plant,Biomass (dedicated),NaN,NaN,8.80,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


We need to understand the shape of the dataset first to understand how many instances and features we are working with. This will help us apply the correct preprocessing steps later on.

In [8]:
df.shape

(13524, 53)

So that's 53 features and over 13,000 instances. Now to use the .info method to examine the column data types and check for any missing values.

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13524 entries, 0 to 13523
Data columns (total 53 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   Old Ref ID                               13524 non-null  int64  
 1   Ref ID                                   11808 non-null  object 
 2   Record Last Updated (dd/mm/yyyy)         13524 non-null  object 
 3   Operator (or Applicant)                  13522 non-null  object 
 4   Site Name                                13521 non-null  object 
 5   Technology Type                          13524 non-null  object 
 6   Storage Type                             2566 non-null   object 
 7   Storage Co-location REPD Ref ID          1518 non-null   object 
 8   Installed Capacity (MWelec)              12340 non-null  object 
 9   Share Community Scheme                   149 non-null    object 
 10  CHP Enabled                              8647 

## Furthering Our Understanding of this Dataset

I want to examine the non-N/A values in columns I may want to move forward with as features. These feature candidates are listed below.

In [10]:
cols = [
    "Offshore Wind Round",
    "FiT Tariff (p/kWh)",
    "CfD Allocation Round",
    "Appeal Refused",
    "RO Banding (ROC/MWh)",
    "CfD Capacity (MW)",
    "Height of Turbines (m)",
    "Turbine Capacity",
    "Under Construction",
    "Storage Type",
    "No. of Turbines",
    "Operational",
    "Solar Site Area (sqm)",
    "Mounting Type for Solar",
    "CHP Enabled",
    "Installed Capacity (MWelec)",
    "Planning Application Submitted",
    "Planning Authority",
    "Site Name",
    "Operator (or Applicant)",
    "County",
    "Region",
    "Record Last Updated (dd/mm/yyyy)",
    "Technology Type",
    "Development Status"
]

for col in cols:
    s = df[col]  # <- the Series
    mask = s.notna() & s.astype("string").str.strip().ne("")
    print(col)
    print(" "*60)
    print(df.loc[mask, col].head(20))
    print("-"*60)

Offshore Wind Round
                                                            
2006    03/01/1900
2007    03/01/1900
2008    03/01/1900
2009    03/01/1900
2010    01/01/1900
2011    01/01/1900
2012    01/01/1900
2013          Demo
2014    03/01/1900
2017    01/01/1900
2018          Demo
2019    02/01/1900
2020    01/01/1900
2021    01/01/1900
2022    01/01/1900
2023    01/01/1900
2024    01/01/1900
2025    01/01/1900
2026    01/01/1900
2027    01/01/1900
Name: Offshore Wind Round, dtype: object
------------------------------------------------------------
FiT Tariff (p/kWh)
                                                            
1014     6.38
1205     6.38
1455     6.38
1731      1.4
3862     7.24
3863     7.24
3866     6.38
3867     3.07
3868     6.38
3870     6.38
3871     6.38
3872    10.96
3874     6.38
3876     6.38
3878     6.38
3879     7.24
3884    10.96
3888     6.38
3893     6.38
3894     6.38
Name: FiT Tariff (p/kWh), dtype: object
-------------------------------------

There is an issue with the "Offshore Wind Round" column. The rounds should be strings like "1", "2", and "3", however at somepoint they have been incorrectly converted to dates. This will corrected downstream.

## Defining the Target Column

## The Target Column Must be Created First

The column we are interested in is the "Planning Permission Granted" column. However the true value is more complicated then a simple yes or no. There are actually many other columns that signify whether planning permission has actaully ben granted or not. For example, a filled "Under Construction" and "Appeal Granted" obviosuly mean that planning permission was granted at some point. There are also cases where the planning permission descision was not actually made (i.e.  "Pending", "Withdrawn", "Revisied", and "Abandoned" projects). We are not interested in these inbetween statuses, and so they will be dropped from our dataset.  

So our plan is to perform some early feature engineering and create a new column called "Target". 

So to be clear, this code:

- Converts Dates: Ensures Planning Permission Granted is a real date object so we can check if it exists.
- Sets Approved (1): Tags all the standard "Yes" statuses.
- Sets Refused (0): Tags all the standard "No" statuses.
- Fixes Abandoned: Specifically looks for rows that are "Abandoned" BUT have a date in the granted column, and sets them to 1.
- Drops the Rest: Removes Pending, Withdrawn, Revised, and the Abandoned projects that were never approved.

In [11]:
# Convert 'Planning Permission Granted' to datetime.
# We need this to check if "Abandoned" projects were actually approved previously.
df["Planning Permission Granted"] = pd.to_datetime(
    df["Planning Permission Granted"], 
    dayfirst=True, 
    errors="coerce"
)

yes_status = [
    "Operational", 
    "Under Construction", 
    "Planning Permission Granted",
    "Planning Permission Expired", 
    "Appeal Granted",
    "Secretary of State - Granted", 
    "Decommissioned"
]

no_status = [
    "Planning Permission Refused", 
    "Appeal Refused", 
    "Secretary of State - Refusal"
]

# Initialising Target Column.
# We start with NaN so we can drop everything that doesn't fit our logic later.
df["Target"] = np.nan

# Apply the Logic

# Rule A: The Explicit YES.
df.loc[df["Development Status"].isin(yes_status), "Target"] = 1

# Rule B: The Explicit NO.
df.loc[df["Development Status"].isin(no_status), "Target"] = 0

# Rule C: The "Abandoned" Logic
# IF Status is 'Abandoned' AND 'Planning Permission Granted' is NOT NULL -> Target = 1
abandoned_approved_mask = (df["Development Status"] == "Abandoned") & (df["Planning Permission Granted"].notnull())
df.loc[abandoned_approved_mask, "Target"] = 1

# Filter the Dataset
# This automatically drops 'Pending', 'Withdrawn', 'Revised', and 'Abandoned-without-grant'
# because their Target is still NaN.
df_clean = df.dropna(subset=["Target"]).copy()

# Convert Target to integer (0 or 1)
df_clean["Target"] = df_clean["Target"].astype(int)

# --- Verification Output ---
print("Shape of cleaned dataset:", df_clean.shape)
print("\nTarget Distribution:")
print(df_clean["Target"].value_counts())
print("\nPercentage Split:")
print(df_clean["Target"].value_counts(normalize=True) * 100)

Shape of cleaned dataset: (10272, 54)

Target Distribution:
Target
1    9101
0    1171
Name: count, dtype: int64

Percentage Split:
Target
1    88.600078
0    11.399922
Name: proportion, dtype: float64


In [12]:
df_clean.drop(columns=[
    "Planning Permission Granted",
    "Operational"
], inplace=True)

We want the target column to be a DataFrame of shape (n, 1). This is important later on. 

In [13]:
y = df_clean[["Target"]].astype(int)
X = df_clean.drop(columns=["Target"]).copy()

## Cleaning the Target Column

Now the target column has been decided, time to give it a quick clean. 

In [14]:
y["Target"] = (
    y["Target"]
    .astype("string")         
    .str.strip()              
    .replace("", pd.NA)       
    .fillna("unknown")        
)

## Observing the Class Balance in Our Target Column

In [15]:
y["Target"].value_counts(normalize=True)

Target
1    0.886001
0    0.113999
Name: proportion, dtype: Float64

In [16]:
df = df_clean

As we can see, the dataset is very unbalanced, so accuracy will not be used as a performance metric. We now have no further use for the following columns and will drop them from our dataset. 

Ideally, to avoid overfitting, we want to maintain an instance to feature number ratio maximum of 1:10. Let's find out roughly what that number should be.

In [17]:
print(f"Max Ideal Number of Features: {X.shape[0]/10}")

Max Ideal Number of Features: 1027.2


## Cleaning the Dataset (Features)

Tasks:

- Resolve all NAs.
- Change columns types were appriopiate.
- Drop duplicates.
- Decide on encoding stratergy.
- Ensure all columns are integers.

## Dropping Unneeded Features

Due to the presence of categorical features we will need to use encoding, which will rapidly increase the number of features. Furthermore, we have over 50 features, not all of which are relevant to our project (i.e. the administration rows). Therefore we need to drop unneeded columns now, simplifying our model and avoiding overfitting. The reasoning for why a column was dropped can be seen below.

In [18]:
drop_features = [
    "Ref ID", # Admin
    "Old Ref ID",
    "Record Last Updated (dd/mm/yyyy)", # Data leak
    "Development Status",
    "Development Status (short)", # Data leak
    "Post Code", # Redundent with county and address
    "X-coordinate", # Unneeded, we have enough address data
    "Y-coordinate", # Unneeded, we have enough address data 
    "Address", # Maybe too granular, plus we have enough 
    "Heat Network Ref", # Admin
    "Appeal Reference",
    "Planning Application Reference",
    "Judicial Review",
    "Secretary of State Reference",
    "Type of Secretary of State Intervention",
    "Planning Application Withdrawn",
    "Share Community Scheme", # This column was empty
    "Planning Permission Refused",
    "Appeal Lodged",
    "Appeal Withdrawn",
    "Appeal Refused",
    "Address", # This information is too granular for our needs, and would make number of features blow up if encoded
    "Appeal Granted",
    "Country", 
    "County",
    "Development Status", # Data leak
    "Secretary of State - Intervened",
    "Storage Co-location REPD Ref ID", # Admin
    "Secretary of State - Refusal", # Data leak
    "Secretary of State - Granted", # Data leak
    "Planning Permission Expired", # Data leak
    "Heat Network Ref", #admin
    "Are they re-applying (New REPD Ref)", # Data leak
    "Under Construction", # Data leak
    "Appeal Refused", # Data leak
    "Record Last Updated (dd/mm/yyyy)" # Data leak, (e.g., record updated yesterday = decision made yesterday)
]

X = X.drop(columns=drop_features, errors="ignore")

X.columns

Index(['Operator (or Applicant)', 'Site Name', 'Technology Type',
       'Storage Type', 'Installed Capacity (MWelec)', 'CHP Enabled',
       'CfD Allocation Round', 'RO Banding (ROC/MWh)', 'FiT Tariff (p/kWh)',
       'CfD Capacity (MW)', 'Turbine Capacity', 'No. of Turbines',
       'Height of Turbines (m)', 'Mounting Type for Solar',
       'Are they re-applying (Old REPD Ref) ', 'Region', 'Planning Authority',
       'Offshore Wind Round', 'Planning Application Submitted',
       'Solar Site Area (sqm)'],
      dtype='object')

To better plan our pre-processing strategy, we need to understand the relative frequency of unique values in a column.

In [19]:
for col in X.columns:
    print("Here the breakdown of column: ", str(col))
    print(X[col].value_counts(dropna=False, normalize=True))
    print("="*60)

Here the breakdown of column:  Operator (or Applicant)
Operator (or Applicant)
Private Developer                     0.026967
Lightsource Renewable Energy          0.014798
AMP Clean Energy                      0.011390
Scottish and Southern Energy (SSE)    0.007009
Renewable Energy Systems (RES)        0.005938
                                        ...   
Balcas Timber                         0.000097
Green Renewable Energy Company        0.000097
Balcas                                0.000097
Unity College                         0.000097
AIPUT Industrial GP Limited           0.000097
Name: proportion, Length: 5323, dtype: float64
Here the breakdown of column:  Site Name
Site Name
Home Farm                                                            0.000584
Manor Farm                                                           0.000487
Gately Moor Reservoir, Redmarshall - Solar Farm & Battery Storage    0.000389
Princess Yachts Limited, Coypool Road - Solar Panels                 0.0

## Checking the True Null Count

In [20]:
def remove_whitespace(df, col):
    df[col] = (
        df[col]
        .astype("string")     
        .str.strip()           
        .replace("", pd.NA)    
        .fillna("unknown")     
    )
    return df

In [21]:
X = remove_whitespace(X, "Offshore Wind Round")

X["Offshore Wind Round"].value_counts(dropna=False, normalize=True)

Offshore Wind Round
unknown       0.994451
02/01/1900    0.001947
03/01/1900    0.001558
01/01/1900    0.001266
Demo          0.000487
STW           0.000292
Name: proportion, dtype: Float64

## Data Splitting and Scaling

We want an 80:20 split between the training and testing datasets. Stratification is on as it is important to maintain the same class balance across datasets.

In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y  
)

## Resolving Nulls

Here is our strategy for handling nulls in specific columns:

1. Financial & Subsidy Columns → Fill with 0
- Reason: Missing values usually means that the project did not get a subsidy. 
* FiT Tariff (p/kWh)
* RO Banding (ROC/MWh)
* CfD Capacity (MW)


2. Physical & Technical Specs → Fill with -1
- Reason: Nulls here are "Not Applicable" for many technologies (e.g., Solar has no turbine height). Filling with "-1" allows the model to separate "N/A" from actual small values (like 5m height).
* Height of Turbines (m)
* Turbine Capacity
* No. of Turbines
* Solar Site Area (sqm)


3. Categorical Columns → Fill with "Unknown"
- Reason: Here we will treats nulls, aka, missing infomation, as its own category. This is because often a blank here is a predictor of planning refusal.
* Region
* Storage Type
* Planning Authority
* Technology Type
* Mounting Type for Solar


4. Ordinal Ranks → Fill with "None" (Map to -1)
- Reason: You need a specific rank that means "Did not participate in any round." Later, your mapping dictionary will convert `"None"` to `-1` or `0`.
* Offshore Wind Round
* CfD Allocation Round


5. Boolean Flags → Fill with 0 (False)
 Reason: If "CHP" is blank, it is standard power (Not CHP). If "Re-applying" is blank, it is a fresh application (Not Re-applying).
* CHP Enabled
* Are they re-applying (Old REPD Ref) 


6. Special Case: Project Size → Fill with Median
- Reason: Every project must have a capacity size. If it is missing, it is a data error. Fill it with the median capacity of its specific `Technology Type` (e.g., fill missing Solar with the median Solar size).
* Installed Capacity (MWelec)


7. BERT Text Inputs → Fill with "" 
- Reason: You are joining these two strings together. If one is NaN, the whole join fails. Filling with an empty string ensures the text cleaner runs smoothly.
* Operator (or Applicant)
* Site Name

In [23]:
zero_fill = [
    "FiT Tariff (p/kWh)", 
    "RO Banding (ROC/MWh)", 
    "CfD Capacity (MW)"
]

neg_one_fill = [
    "Height of Turbines (m)", 
    "Turbine Capacity", 
    "No. of Turbines", 
    "Solar Site Area (sqm)"
]

unknown_fill = [
    "Region", 
    "Storage Type", 
    "Planning Authority", 
    "Technology Type", 
    "Mounting Type for Solar"
]

none_fill = [
    "Offshore Wind Round", 
    "CfD Allocation Round"
]

median_fill = [
    "Installed Capacity (MWelec)"
]

empty_string_fill = [
    "Operator (or Applicant)", 
    "Site Name"
]

In [24]:
# 1. Zero Fill
for col in zero_fill:
    X_train[col] = pd.to_numeric(X_train[col], errors='coerce').fillna(0)
    X_test[col] = pd.to_numeric(X_test[col], errors='coerce').fillna(0)

# 2. Negative One Fill
for col in neg_one_fill:
    X_train[col] = pd.to_numeric(X_train[col], errors='coerce').fillna(-1)
    X_test[col] = pd.to_numeric(X_test[col], errors='coerce').fillna(-1)

# 3. Unknown Fill (String)
for col in unknown_fill:
    X_train[col] = X_train[col].fillna("Unknown") 
    X_test[col] = X_test[col].fillna("Unknown")

# 4. None Fill (String, for later mapping)
for col in none_fill:
    X_train[col] = X_train[col].fillna("None")
    X_test[col] = X_test[col].fillna("None")

# 5. Median Fill (Smart Grouping)
for col in median_fill:
    # A. Force numeric (coerces errors to NaN). Ensure numeric
    X_train[col] = pd.to_numeric(X_train[col], errors="coerce")
    X_test[col] = pd.to_numeric(X_test[col], errors="coerce")
    
    # B. Fill with Tech-specific median. Fill with the median of the SPECIFIC Technology Type. (e.g. Missing Solar gets Solar median, missing Wind gets Wind median)
    train_group_medians = X_train.groupby("Technology Type")[col].median()
    train_global_median = X_train[col].median()

    # fill train
    X_train[col] = X_train[col].fillna(X_train["Technology Type"].map(train_group_medians))
    X_train[col] = X_train[col].fillna(train_global_median)
    
    # fill test
    X_test[col] = X_test[col].fillna(X_test["Technology Type"].map(train_group_medians))
    X_test[col] = X_test[col].fillna(train_global_median)

# 6. Empty String Fill (for Text Joining). Fallback: If a technology is unique and has no median, use global median
for col in empty_string_fill:
    X_train[col] = X_train[col].fillna("")
    X_test[col] = X_test[col].fillna("")

# Here we're quickly making a separate column that compliments the "CHP Enabled" column. This will be a "Missing/Unknown" flag that will work alongside our CHP column during machine learning.
X_train["CHP Enabled_unknown"] = (
    X_train["CHP Enabled"].isna() |
    X_train["CHP Enabled"].astype("string").str.strip().str.lower().isin(["nan", "unknown", "none", ""])
).astype(int)

X_test["CHP Enabled_unknown"] = (
    X_test["CHP Enabled"].isna() |
    X_test["CHP Enabled"].astype("string").str.strip().str.lower().isin(["nan", "unknown", "none", ""])
).astype(int)

Now we need to check if all the null values have been addressed before moving on. 

In [25]:
missing_values_count = X_train.isnull().sum()
number_of_rows = X_train.shape[0]
(missing_values_count[0:] / number_of_rows * 100).round(2)

Operator (or Applicant)                  0.00
Site Name                                0.00
Technology Type                          0.00
Storage Type                             0.00
Installed Capacity (MWelec)              0.00
CHP Enabled                             38.10
CfD Allocation Round                     0.00
RO Banding (ROC/MWh)                     0.00
FiT Tariff (p/kWh)                       0.00
CfD Capacity (MW)                        0.00
Turbine Capacity                         0.00
No. of Turbines                          0.00
Height of Turbines (m)                   0.00
Mounting Type for Solar                  0.00
Are they re-applying (Old REPD Ref)     94.40
Region                                   0.00
Planning Authority                       0.00
Offshore Wind Round                      0.00
Planning Application Submitted           1.76
Solar Site Area (sqm)                    0.00
CHP Enabled_unknown                      0.00
dtype: float64

In [26]:
missing_values_count = X_test.isnull().sum()
number_of_rows = X_test.shape[0]
(missing_values_count[0:] / number_of_rows * 100).round(2)

Operator (or Applicant)                  0.00
Site Name                                0.00
Technology Type                          0.00
Storage Type                             0.00
Installed Capacity (MWelec)              0.00
CHP Enabled                             38.25
CfD Allocation Round                     0.00
RO Banding (ROC/MWh)                     0.00
FiT Tariff (p/kWh)                       0.00
CfD Capacity (MW)                        0.00
Turbine Capacity                         0.00
No. of Turbines                          0.00
Height of Turbines (m)                   0.00
Mounting Type for Solar                  0.00
Are they re-applying (Old REPD Ref)     94.31
Region                                   0.00
Planning Authority                       0.00
Offshore Wind Round                      0.00
Planning Application Submitted           1.70
Solar Site Area (sqm)                    0.00
CHP Enabled_unknown                      0.00
dtype: float64

## Resolving Duplicates

First we need to count the duplicates that occur in all columns.

In [27]:
from typing import Optional, Union, Sequence

def count_dup_rows(dataframe: pd.DataFrame, 
                   column_name: Union[str, Sequence[str]] = None) -> None:
    """
    Counts and prints the number and percentages of duplicate rows in a dataframe.

    :param dataframe: Pandas DataFrame to analyse.
    :param column_name: Column used to identify duplicate rows.
    :return: None
    """
    print("="*90)
    print(f"Found {dataframe.duplicated(subset=None, keep='first').sum()} duplicate rows")
    print(
        f"This dataframe has {dataframe.shape[0]} rows btw\n"
          f"Therefore {dataframe.duplicated(subset=column_name, keep='first').sum() * 100 / dataframe.shape[0]}% of rows"
        f" are dups"
          )
    # When you don't specify the subset argument, Pandas considers a row to be a duplicate if all of its values
    # across all of its columns are identical to a previous row

In [28]:
# for col in X_train.columns:
print("The number of duplicates in the train dataset: ", count_dup_rows(X_train))
print("The number of duplicates in the test dataset: ", count_dup_rows(X_test))

Found 4 duplicate rows
This dataframe has 8217 rows btw
Therefore 0.04867956675185591% of rows are dups
The number of duplicates in the train dataset:  None
Found 0 duplicate rows
This dataframe has 2055 rows btw
Therefore 0.0% of rows are dups
The number of duplicates in the test dataset:  None


Duplicate rows are removed before further preprocessing.

In [29]:
# Drop duplicates in X_train and keep corresponding y_train rows.
X_train = X_train.drop_duplicates(keep='first')
y_train = y_train.loc[X_train.index]

# Drop duplicates in X_test and keep corresponding y_test rows.
X_test = X_test.drop_duplicates(keep='first')
y_test = y_test.loc[X_test.index]

In [30]:
# For col in X_train.columns:
print("The number of duplicates in the train dataset: ", count_dup_rows(X_train))
print("The number of duplicates in the test dataset: ", count_dup_rows(X_test))

Found 0 duplicate rows
This dataframe has 8213 rows btw
Therefore 0.0% of rows are dups
The number of duplicates in the train dataset:  None
Found 0 duplicate rows
This dataframe has 2055 rows btw
Therefore 0.0% of rows are dups
The number of duplicates in the test dataset:  None


## Fixing the "Offshore Wind Round" column

Remember the issue with the "Offshore Wind Round" column mentioned earlier, where the rounds that should be strings like "1", "2", and "3" look like dates? Now we will address that. Below we have created a function that turns these dates back into string numbers.

In [31]:
def clean_excel_round_artifacts(X, col, inplace=True):
    if not inplace:
        X = X.copy()

    if col not in X.columns:
        raise KeyError(f"Column '{col}' not found in DataFrame.")

    s = X[col]

    dt = pd.to_datetime(s, errors="coerce", dayfirst=True)

    # only treat the weird Excel date artifacts (Jan 1900) as rounds
    mask = dt.notna() & (dt.dt.year == 1900) & (dt.dt.month == 1)

    # convert 1900-01-01 -> 1, 1900-01-02 -> 2, etc.
    round_num = (dt - pd.Timestamp("1899-12-31")).dt.days.astype("Int64")

    out = s.astype("string").str.strip()
    out = out.mask(mask, round_num.astype("string"))  # Replaces the date artifacts with string numbers like "1","2", "3"
    out = out.replace("", pd.NA).fillna("unknown")

    X[col] = out
    return X

In [32]:
X_train = clean_excel_round_artifacts(X_train, "Offshore Wind Round")
X_test = clean_excel_round_artifacts(X_test, "Offshore Wind Round")

I then verify that the transformation was applied correctly.

In [33]:
X_train["Offshore Wind Round"].value_counts(dropna=False, normalize=True).head(10)

Offshore Wind Round
unknown    0.994399
2          0.001948
3          0.001461
1          0.001218
Demo       0.000609
STW        0.000365
Name: proportion, dtype: Float64

## Ordinal Mapping for the "Offshore Wind Round" Column

This is an ideal time to encode the "Offshore Wind Round" column. Here is how we will map things:

Demo → 0: It represents the "starting point" or small-scale testing before the big rounds.
Round 1 → 1
Round 2 → 2
STW → 2: (STW means "Scottish Territorial Waters", and this happens roughly alongside rounds 2/3. Here we're treating it as "2" to be safe).
Round 3 → 3
Extensions → Extensions are usually huge.

In [34]:
rank_map = {
   "Demo": 0,
   "1": 1,
   "2": 2,
   "STW": 2,
   "3": 3,
   "Extension": 3 
}

X_train["Round_Rank"] = X_train["Offshore Wind Round"].map(rank_map)
X_test["Round_Rank"] = X_test["Offshore Wind Round"].map(rank_map)

# Filling NaNs with -1 (meaning "No Offshore Wind Round") as we want to keep them distinct.
X_train["Round_Rank"] = X_train["Round_Rank"].fillna(-1)
X_test["Round_Rank"] = X_test["Round_Rank"].fillna(-1)

## Ordinal Mapping for the "CfD Allocation Round" Column

Here's our justification for how we have decided to order the following:

- Investment Contract (0): These were the "FID Enabling" contracts signed before the first auction. They are the "Round 0" of this system.
- AR1 - AR6: Standard sequential order.
- NaN (-1): Most projects don't have a CfD. It is crucial to distinguish "No CfD" (-1) from "Early CfD" (0).

In [35]:
cfd_rank_map = {
    "Investment Contract": 0, 
    "AR1": 1,
    "AR1 ": 1,                # This is here to handle whitespace variant we spotted.
    "AR2": 2,
    "AR3": 3,
    "AR4": 4,
    "AR5": 5,
    "AR6": 6
}

# Apply the mapping.
X_train["CfD_Round_Rank"] = X_train["CfD Allocation Round"].map(cfd_rank_map)
X_test["CfD_Round_Rank"] = X_test["CfD Allocation Round"].map(cfd_rank_map)

# Filling NaNs with -1 (meaning "No CfD") as we want to keep them distinct.
X_train["CfD_Round_Rank"] = X_train["CfD_Round_Rank"].fillna(-1)
X_test["CfD_Round_Rank"] = X_test["CfD_Round_Rank"].fillna(-1)

## Fixing the "Mounting Type for Solar" Column

The "Mounting Type for Solar" column can have multiple categories per row, so we must separate them each into different columns according to the ";" delimiter. Here's how the column currently looks.

In [36]:
X_train["Mounting Type for Solar"].head(15)

2266     Unknown
11305       Roof
4545     Unknown
9321        Roof
7604      Ground
7886     Unknown
10977       Roof
8063        Roof
12184       Roof
2987     Unknown
1130      Ground
5078     Unknown
7800        Roof
8925     Unknown
1270      Ground
Name: Mounting Type for Solar, dtype: object

In [37]:
def split_delimited(column: pd.Series, delimiter: str = ";") -> pd.Series:
    """
    Turns 'a; b; c' -> ['a','b','c'] (per row).
    - trims whitespace
    - lowercases
    - drops empty tokens and 'unknown'
    """
    s = column.astype("string").fillna("").str.strip().str.lower()

    split_col = s.str.split(delimiter)

    split_col = split_col.apply(
        lambda lst: [
            t.strip() for t in lst
            if t and t.strip() and t.strip().lower() != "unknown"
        ]
    )
    return split_col

In [38]:
X_train["Mounting Type for Solar_tokens"] = split_delimited(X_train["Mounting Type for Solar"], ";")
X_test["Mounting Type for Solar_tokens"] = split_delimited(X_test["Mounting Type for Solar"], ";")

In [39]:
X_train["Mounting Type for Solar_tokens"].head(15)

2266           []
11305      [roof]
4545           []
9321       [roof]
7604     [ground]
7886           []
10977      [roof]
8063       [roof]
12184      [roof]
2987           []
1130     [ground]
5078           []
7800       [roof]
8925           []
1270     [ground]
Name: Mounting Type for Solar_tokens, dtype: object

## Cleaning/Feature engineering the "CHP Enabled" column

The column is a bit of a mess. Due to inconsistent capitalisation, there are multiple versions of yes and no. 

CHP Enabled freq distribution:
- No: 0.611136
- unknown: 0.360618
- Yes: 0.027728
- yes: 0.00037
- no: 0.000074
- NO: 0.000074

So the column will be cleaned, consolidated and simplified to be a simple "yes" (symbolised by 1) and "no" (symbolised by 0).

This is how the column looks now.

In [40]:
X_train["CHP Enabled"].head(15)

2266     NaN
11305     No
4545      No
9321      No
7604      No
7886      No
10977     No
8063      No
12184     No
2987     NaN
1130     NaN
5078     Yes
7800      No
8925      No
1270     NaN
Name: CHP Enabled, dtype: object

In [41]:
X_train["CHP Enabled"] = X_train["CHP Enabled"].str.lower().str.strip()
X_test["CHP Enabled"] = X_test["CHP Enabled"].str.lower().str.strip()

In [42]:
X_train["CHP Enabled"].head(15)

2266     NaN
11305     no
4545      no
9321      no
7604      no
7886      no
10977     no
8063      no
12184     no
2987     NaN
1130     NaN
5078     yes
7800      no
8925      no
1270     NaN
Name: CHP Enabled, dtype: object

In [43]:
X_train["CHP Enabled"] = (X_train["CHP Enabled"] == "yes").astype(int)
X_test["CHP Enabled"] = (X_test["CHP Enabled"] == "yes").astype(int)

In [44]:
X_train["CHP Enabled"].head(15)

2266     0
11305    0
4545     0
9321     0
7604     0
7886     0
10977    0
8063     0
12184    0
2987     0
1130     0
5078     1
7800     0
8925     0
1270     0
Name: CHP Enabled, dtype: int64

Let's compare this new column with the "CHP Enabled_unknown" flag column we quickly made earlier. If both columns contain "1" then clearly something has gone wrong.

In [45]:
print(X_train[["CHP Enabled", "CHP Enabled_unknown"]].head(15))
print(X_test[["CHP Enabled", "CHP Enabled_unknown"]].head(15))

       CHP Enabled  CHP Enabled_unknown
2266             0                    1
11305            0                    0
4545             0                    0
9321             0                    0
7604             0                    0
7886             0                    0
10977            0                    0
8063             0                    0
12184            0                    0
2987             0                    1
1130             0                    1
5078             1                    0
7800             0                    0
8925             0                    0
1270             0                    1
       CHP Enabled  CHP Enabled_unknown
9526             0                    0
2942             0                    1
10959            0                    0
502              0                    0
2236             0                    1
6080             0                    0
8572             0                    0
9380             0                    0


## General EDA

Now things are less cluttered , we can inspect the remaining dataset more carefully via simple exploratory data analysis. This will aid our future cleaning and preprocessing strategies. 

In [46]:
X_train["Height of Turbines (m)"].value_counts(dropna=False, normalize=True)

Height of Turbines (m)
-1.0      0.941191
 149.9    0.004383
 100.0    0.004018
 200.0    0.003409
 150.0    0.003287
            ...   
 170.0    0.000122
 85.0     0.000122
 73.0     0.000122
 113.0    0.000122
 48.0     0.000122
Name: proportion, Length: 122, dtype: float64

In [47]:
for col in X_train.columns:
    try:
        number_counts = X_train[col].nunique()
        print(f"Column: {col}, Number of unique values: {number_counts}")
    except TypeError as e:
        print(f"Error with column: {col} - {e}")

Column: Operator (or Applicant), Number of unique values: 4463
Column: Site Name, Number of unique values: 7957
Column: Technology Type, Number of unique values: 25
Column: Storage Type, Number of unique values: 15
Column: Installed Capacity (MWelec), Number of unique values: 697
Column: CHP Enabled, Number of unique values: 2
Column: CfD Allocation Round, Number of unique values: 9
Column: RO Banding (ROC/MWh), Number of unique values: 10
Column: FiT Tariff (p/kWh), Number of unique values: 7
Column: CfD Capacity (MW), Number of unique values: 143
Column: Turbine Capacity, Number of unique values: 108
Column: No. of Turbines, Number of unique values: 77
Column: Height of Turbines (m), Number of unique values: 122
Column: Mounting Type for Solar, Number of unique values: 13
Column: Are they re-applying (Old REPD Ref) , Number of unique values: 387
Column: Region, Number of unique values: 14
Column: Planning Authority, Number of unique values: 380
Column: Offshore Wind Round, Number of 

In [48]:
for col in X_train.columns:
    print(f"No. of unique values in {col}: {X_train[col].count}")

No. of unique values in Operator (or Applicant): <bound method Series.count of 2266                      Greencoat
11305      St Johns Beaumont School
4545             Mssrs R Miller Ltd
9321                      Edbro Plc
7604     Sands Of Luce Holiday Park
                    ...            
1548         FIM Solar Distribution
808                            SUEZ
1226            Gamma Solar Limited
11195                  Truro School
10732                   RES Limited
Name: Operator (or Applicant), Length: 8213, dtype: object>
No. of unique values in Site Name: <bound method Series.count of 2266                     Brockaghboy Wind Farm - extension
11305    St Johns Beaumont School, Priest Hill - Solar ...
4545                                          Arkleby Mill
9321                    Edbro, Lever Street - Solar Panels
7604                     Sandmill, Sandhead - Solar Panels
                               ...                        
1548                                Congresbur

In [49]:
print("="*90)
print("Here is the Dataframe information")
print("")
print("Shape:")
print(df.shape)
print("-"*90)
print("")
print("")
print("Head:")
print(X_train.head())
print("-" * 90)
print("")
print("Tail:")
print(X_train.tail())
print("-" * 90)
print("")
print("Null Count: ")
print(X_train.isnull().sum())
print("")
print("Now as percentages:")
missing_values_count = X_train.isnull().sum()
number_of_rows = X_train.shape[0]
print((missing_values_count[0:] / number_of_rows * 100).round(2))
total_cells = np.prod(X_train.shape)
total_missing = missing_values_count.sum()
percent_missing = (total_missing / total_cells) * 100
percent_missing = round(percent_missing, 2)
print(f"This is the percentage of cells that are empty: {percent_missing}%")
print("-" * 90)
print("")
print("Describe: ")
print(X_train.describe(percentiles=None, include="all").T)
print("-" * 90)
print("")
print("Info:")
print(X_train.info(verbose=True, memory_usage=True))
print("="*90)

Here is the Dataframe information

Shape:
(10272, 52)
------------------------------------------------------------------------------------------


Head:
          Operator (or Applicant)  \
2266                    Greencoat   
11305    St Johns Beaumont School   
4545           Mssrs R Miller Ltd   
9321                    Edbro Plc   
7604   Sands Of Luce Holiday Park   

                                               Site Name      Technology Type  \
2266                   Brockaghboy Wind Farm - extension         Wind Onshore   
11305  St Johns Beaumont School, Priest Hill - Solar ...  Solar Photovoltaics   
4545                                        Arkleby Mill  Anaerobic Digestion   
9321                  Edbro, Lever Street - Solar Panels  Solar Photovoltaics   
7604                   Sandmill, Sandhead - Solar Panels  Solar Photovoltaics   

      Storage Type  Installed Capacity (MWelec)  CHP Enabled  \
2266       Unknown                        10.00            0   
11305    

Next is to group columns appropriately for future cleaning and preprocessing steps.

## Groups for Cleaning

In [50]:
num_cols = [
    "Installed Capacity (MWelec)", 
    "FiT Tariff (p/kWh)", 
    "RO Banding (ROC/MWh)", 
    "CfD Capacity (MW)", 
    "Turbine Capacity", 
    "Height of Turbines (m)", 
    "No. of Turbines", 
    "Solar Site Area (sqm)"
]

date_cols = [
    "Record Last Updated (dd/mm/yyyy)", 
    "Appeal Refused", 
    "Under Construction", 
    "Operational"
]

text_cols = [
    "Technology Type", 
    "Region", 
    "Planning Authority", 
    "Operator (or Applicant)", 
    "Site Name", 
    "Storage Type", 
    "Mounting Type for Solar"
]

bool_cols = [
    "CHP Enabled",
    "CHP Enabled_unknown",
    "Are they re-applying (Old REPD Ref) ",
    "Round_Rank", # Feature engineered 
    "CfD_Round_Rank" # Feature engineered
]

## Groups for Encoding

In [51]:
columns_for_one_hot_encoding = [
    "Region",
    "Technology Type", 
    "Storage Type",
    "Planning Authority"
]

columns_for_multi_hot_encoding_mapping = [
    "Mounting Type for Solar"
]

## Cleaning Whitespace

Cleaning all string rows of white space, capitalising them, and removes non-printing ASCII characters from the string columns.

In [52]:
for col in text_cols:
    X_train[col] = X_train[col].astype(str).str.lower().str.strip()
    X_test[col] = X_test[col].astype(str).str.lower().str.strip()
    print("-"*90)
    print(f"All whitespace from the following columns removed: {col}")

------------------------------------------------------------------------------------------
All whitespace from the following columns removed: Technology Type
------------------------------------------------------------------------------------------
All whitespace from the following columns removed: Region
------------------------------------------------------------------------------------------
All whitespace from the following columns removed: Planning Authority
------------------------------------------------------------------------------------------
All whitespace from the following columns removed: Operator (or Applicant)
------------------------------------------------------------------------------------------
All whitespace from the following columns removed: Site Name
------------------------------------------------------------------------------------------
All whitespace from the following columns removed: Storage Type
-----------------------------------------------------------

In [53]:
missing_values_count = X_train.isnull().sum()
number_of_rows = X_train.shape[0]
(missing_values_count[0:] / number_of_rows * 100).round(2)

Operator (or Applicant)                  0.00
Site Name                                0.00
Technology Type                          0.00
Storage Type                             0.00
Installed Capacity (MWelec)              0.00
CHP Enabled                              0.00
CfD Allocation Round                     0.00
RO Banding (ROC/MWh)                     0.00
FiT Tariff (p/kWh)                       0.00
CfD Capacity (MW)                        0.00
Turbine Capacity                         0.00
No. of Turbines                          0.00
Height of Turbines (m)                   0.00
Mounting Type for Solar                  0.00
Are they re-applying (Old REPD Ref)     94.40
Region                                   0.00
Planning Authority                       0.00
Offshore Wind Round                      0.00
Planning Application Submitted           1.77
Solar Site Area (sqm)                    0.00
CHP Enabled_unknown                      0.00
Round_Rank                        

## Feature Engineering

## Condensing the "Technology Type column"

As observed in our earlier EDA, there are very similar categories which mean essentially the same thing for our purposes. For example, "Large Hydro" has only ~30 examples and "Small Hydro" has ~150. This imbalance makes it hard for a model to learn a stable pattern for "Large Hydro" alone, so it is better to combine them. Furthermore our analysis of our target variable shows they behave very similarly:

- Small Hydro: ~93% Approval Rate
- Large Hydro: ~96% Approval Rate

Since they have the same "signal" (both are highly likely to be approved) and one is very small, keeping them separate adds noise without adding value.
So we will clean up all the tiny categories, not just Hydro. Please see the mapping below.

Note that we are not grouping "Wind Onshore" and "Wind Offshore" as these are obviously very different. They also behave differently according to our target column.

- Wind Offshore: ~96% Approval Rate (Government priority).
- Wind Onshore: ~68% Approval Rate (Controversial, often refused).

In [54]:
tech_map = {
    # 1. Grouping all Hydro.
    "Small Hydro": "Hydro",
    "Large Hydro": "Hydro",
    "Pumped Storage Hydroelectricity": "Hydro",
    
    # 2. Grouping all Biomass/Waste.
    "Biomass (dedicated)": "Biomass",
    "Biomass (co-firing)": "Biomass",
    "Sewage Sludge Digestion": "Biomass", 
    
    # 3. Grouping Marine Technologies.
    "Tidal Stream": "Marine",
    "Tidal Lagoon": "Marine",
    "Shoreline Wave": "Marine",
    
    # 4. Grouping Emerging Tech.
    "Hydrogen": "Emerging Tech",
    "Fuel Cell (Hydrogen)": "Emerging Tech",
    "Compressed Air Energy Storage": "Emerging Tech",
    "Liquid Air Energy Storage": "Emerging Tech",
    "Geothermal": "Emerging Tech",
    "Hot Dry Rocks (HDR)": "Emerging Tech",
    "Flywheels": "Emerging Tech"
}

# Apply the mapping.
X_train["Technology Type"] = X_train["Technology Type"].replace(tech_map)
X_test["Technology Type"] = X_test["Technology Type"].replace(tech_map)

## Date Feature Engineering

Dates by themselves are meaningless for machine learning, so we need to create new features based on them for our model downstream. In this case, "Planning Application Submitted" is the only date column we can really use for prediction, as all the other date columns like "Planning Permission Granted" happen after the decision is made and thus introduce data leakage.

There are a few key features we will be creating from the "Planning Application Submitted" column.
1. Application_Year
2. Application_Month
3. PlanningApp_missing 

Why Application_Year and Application_Month columns? Well planning approval rates can change drastically based on government policy. For example in 2015/2016 the UK government cut subsidies for onshore wind and solar, causing approval rates to drop. Or there might be administrative cycles. Again, for example applications submitted right before the end of the financial year (March) or calendar year (December) might be rushed or treated differently.

In [55]:
# Training Dataset.
dt_train = pd.to_datetime(X_train["Planning Application Submitted"], errors="coerce", dayfirst=True)
year_train = dt_train.dt.year
month_train = dt_train.dt.month

year_med = year_train.median()         
month_fill = int(month_train.mode().iloc[0]) if month_train.notna().any() else 0 

X_train["PlanningApp_missing"] = dt_train.isna().astype("int8")
X_train["Application_Year"]  = year_train.fillna(year_med).astype("int64")
X_train["Application_Month"] = month_train.fillna(month_fill).astype("int64")

# Testing Dataset.
dt_test = pd.to_datetime(X_test["Planning Application Submitted"], errors="coerce", dayfirst=True)
X_test["PlanningApp_missing"] = dt_test.isna().astype("int8")
X_test["Application_Year"]  = dt_test.dt.year.fillna(year_med).astype("int64")
X_test["Application_Month"] = dt_test.dt.month.fillna(month_fill).astype("int64")

I then verify the engineered date features.

In [56]:
X_train["Application_Year"].head(15)

2266     2014
11305    2024
4545     2015
9321     2023
7604     2022
7886     2022
10977    2024
8063     2022
12184    2024
2987     2011
1130     2012
5078     2013
7800     2022
8925     2023
1270     2011
Name: Application_Year, dtype: int64

In [57]:
X_train["Application_Month"].head(15)

2266      6
11305     5
4545      7
9321      4
7604      4
7886      6
10977     2
8063      7
12184    10
2987      1
1130      8
5078      4
7800      6
8925      1
1270      2
Name: Application_Month, dtype: int64

In [58]:
print(X_train["Application_Year"].isnull().sum())
print(X_train["Application_Month"].isnull().sum())

0
0


In [59]:
print(X_test["Application_Year"].isnull().sum())
print(X_test["Application_Month"].isnull().sum())

0
0


We can finally drop the "Planning Application Submitted" column now these features have been created. Dropping these columns also removes its nulls values from our datasets.

In [60]:
X_train = X_train.drop("Planning Application Submitted", axis=1)
X_test = X_test.drop("Planning Application Submitted", axis=1)

These new feature names are added to the existing date feature list.

In [61]:
date_cols.extend(
    [
        "Application_Year", 
        "Application_Month"]
) 

## One Hot Encoding

We found that one hot encoding of these columns caused a specific problem. This encoding of this many categories meant that certain categories appeared in the training set, but not in the test set. Due to the following lines of code; 

X_train = pd.get_dummies(X_train, columns=cols, drop_first=True, dtype=int)
X_test  = pd.get_dummies(X_test,  columns=cols, drop_first=True, dtype=int)

Each slit received dummy columns only for the categories present in that split. For example, say if no test rows had “Technology Type”: “flywheels”, then “Technology Type”: “flywheels” won’ be created in the test matrix. The following code uses the .get_dummies method to force the test columns to match train’s columns, thus fixing the issue. 

In [62]:
X_train = pd.get_dummies(X_train, columns=columns_for_one_hot_encoding, drop_first=True, dtype=int)
X_test  = pd.get_dummies(X_test,  columns=columns_for_one_hot_encoding, drop_first=True, dtype=int)

X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

## Multi-Hot Encoding

Now for multi-hot encoding.

In [63]:
def multihot_fit_transform(train_tokens: pd.Series, prefix: str):
    """
    Fit on train token lists and return:
    - train multi-hot DF
    - fitted binarizer (for transform on val/test)
    """
    mlb = MultiLabelBinarizer()
    X_train_mh = mlb.fit_transform(train_tokens)
    cols = [f"{prefix}{c}" for c in mlb.classes_]
    train_df = pd.DataFrame(X_train_mh, columns=cols, index=train_tokens.index)
    return train_df, mlb

def multihot_transform(tokens: pd.Series, mlb: MultiLabelBinarizer, prefix: str):
    """
    Transform val/test token lists using train-fitted mlb.
    """
    X_mh = mlb.transform(tokens)
    cols = [f"{prefix}{c}" for c in mlb.classes_]
    return pd.DataFrame(X_mh, columns=cols, index=tokens.index)

In [64]:
for col in columns_for_multi_hot_encoding_mapping:

    train_mh, mlb = multihot_fit_transform(X_train[f"{col}_tokens"], f"{col}_")
    test_mh = multihot_transform(X_test[f"{col}_tokens"], mlb, f"{col}_")
    
    X_train = pd.concat([X_train.drop(columns=[col, f"{col}_tokens"]), train_mh], axis=1)
    X_test  = pd.concat([X_test.drop(columns=[col, f"{col}_tokens"]), test_mh], axis=1)

I perform a final missing-value check after encoding.

In [65]:
missing_values_count = X_train.isnull().sum()
number_of_rows = X_train.shape[0]
(missing_values_count[0:] / number_of_rows * 100).round(2)

Operator (or Applicant)                  0.0
Site Name                                0.0
Installed Capacity (MWelec)              0.0
CHP Enabled                              0.0
CfD Allocation Round                     0.0
                                        ... 
Planning Authority_york                  0.0
Mounting Type for Solar_floating         0.0
Mounting Type for Solar_ground           0.0
Mounting Type for Solar_ground & roof    0.0
Mounting Type for Solar_roof             0.0
Length: 437, dtype: float64

## Binary Mapping

### The "Are they re-applying (Old REPD Ref) " column

This is a clear yes or no question. If the value was blank it just means "no", they are not re-applying. So we will convert this to 1 if the cell has a value, and 0 if it is NaN.

In [66]:
X_train["Are they re-applying (Old REPD Ref) "] = X_train["Are they re-applying (Old REPD Ref) "].notna().astype(int)
X_test["Are they re-applying (Old REPD Ref) "] = X_test["Are they re-applying (Old REPD Ref) "].notna().astype(int)

# Check the result.
print(f"Re-applications found: {X_train["Are they re-applying (Old REPD Ref) "].sum()}")

Re-applications found: 460


## BERT Encoding

We decided to concatenate the “Site Name” and “Operator” columns into a single new column. This reduces the number of features created post embedding (now we only create 768 new columns), thus keeping our total number of features under the ideal maximum to avoid overfitting due to dataset shape. Combining these two features is also plausible due to their inherent similarities:

- "Site Name" = The "What"
- “Operator” = The "Who"

Note: The "Target Encode Planning Authority” column was not encoded. It was just kept as a single numerical feature. 

## Creating a New Column for BERT Encoding via Concatenation

In [67]:
def tagged_concat(row, cols):
    parts = []
    for c in cols:
        val = row.get(c)
        if pd.notna(val) and str(val).strip():
            parts.append(f"[{c}] {val}")
    return " ".join(parts)

groups = {
    "TEXT_OPERATOR_AND_SITE_NAME": ["Operator (or Applicant)", 
                   "Site Name"]
}

In [68]:
for new_col, cols in groups.items():
    X_train[new_col] = X_train.apply(lambda r: tagged_concat(r, cols), axis=1)
    X_test[new_col] = X_test.apply(lambda r: tagged_concat(r, cols), axis=1)

In [69]:
columns_for_bert = [
    "TEXT_OPERATOR_AND_SITE_NAME"
]

I verify that the combined text column was created correctly.

In [70]:
X_train["TEXT_OPERATOR_AND_SITE_NAME"].head(5)

2266     [Operator (or Applicant)] greencoat [Site Name...
11305    [Operator (or Applicant)] st johns beaumont sc...
4545     [Operator (or Applicant)] mssrs r miller ltd [...
9321     [Operator (or Applicant)] edbro plc [Site Name...
7604     [Operator (or Applicant)] sands of luce holida...
Name: TEXT_OPERATOR_AND_SITE_NAME, dtype: object

## Setting up the BERT Encoder

In [71]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from transformers import AutoTokenizer, AutoModel

import time
from datetime import datetime
from zoneinfo import ZoneInfo

In [72]:
MODEL_NAME = "bert-base-uncased" 

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def embed_column_sbert(
    df: pd.DataFrame,
    text_col: str,
    batch_size: int = 64,
    pooling: str = "mean",   
) -> np.ndarray:

    texts = df[text_col].fillna("").astype(str).tolist()

    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]

        batch = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True
        )
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.no_grad():
            outputs = model(**batch)

        if pooling == "pooler":
            embs = outputs.pooler_output  
        else:
            # mean pooling with attention mask
            last = outputs.last_hidden_state  
            mask = batch["attention_mask"].unsqueeze(-1)  
            embs = (last * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)

        all_embs.append(embs.detach().cpu().numpy())

    return np.vstack(all_embs)  

I inspect an example input before generating embeddings.

In [73]:
X_train[columns_for_bert].iloc[0].to_dict()

{'TEXT_OPERATOR_AND_SITE_NAME': '[Operator (or Applicant)] greencoat [Site Name] brockaghboy wind farm - extension'}

## Generating Text Embeddings

In [74]:
def bert_encoding_tracker(columns_for_bert, df):
    start = time.perf_counter()
    total = len(columns_for_bert)
    
    for i, col in enumerate(columns_for_bert, start=1):
        embeddings_array = embed_column_sbert(df, col)
    
        emb_col_names = [f"{col}_emb_{j}" for j in range(embeddings_array.shape[1])]
        emb_df = pd.DataFrame(embeddings_array, columns=emb_col_names, index=df.index)
        df = pd.concat([df, emb_df], axis=1)
    
        elapsed = time.perf_counter() - start
        m, s = divmod(elapsed, 60)
        print(f"Epoch {i}/{total} done ({col}) | elapsed: {int(m):02d}:{s:04.1f}")
    
    return df

In [75]:
X_train = bert_encoding_tracker(columns_for_bert, X_train)

Epoch 1/1 done (TEXT_OPERATOR_AND_SITE_NAME) | elapsed: 03:19.4


Checking the result...

In [76]:
print(f"New DataFrame Shape: {X_train.shape}")

New DataFrame Shape: (8213, 1206)


In [77]:
X_train.head()

,Operator (or Applicant),Site Name,Installed Capacity (MWelec),CHP Enabled,CfD Allocation Round,RO Banding (ROC/MWh),FiT Tariff (p/kWh),CfD Capacity (MW),Turbine Capacity,No. of Turbines,...,TEXT_OPERATOR_AND_SITE_NAME_emb_758,TEXT_OPERATOR_AND_SITE_NAME_emb_759,TEXT_OPERATOR_AND_SITE_NAME_emb_760,TEXT_OPERATOR_AND_SITE_NAME_emb_761,TEXT_OPERATOR_AND_SITE_NAME_emb_762,TEXT_OPERATOR_AND_SITE_NAME_emb_763,TEXT_OPERATOR_AND_SITE_NAME_emb_764,TEXT_OPERATOR_AND_SITE_NAME_emb_765,TEXT_OPERATOR_AND_SITE_NAME_emb_766,TEXT_OPERATOR_AND_SITE_NAME_emb_767
2266,greencoat,brockaghboy wind farm - extension,10.00,0,None,0.0,0.0,0.0,3.0,4.0,...,0.125861,-0.162719,0.076838,-0.494589,0.150260,-0.441413,-0.117774,-0.017338,-0.226208,-0.029284
11305,st johns beaumont school,"st johns beaumont school, priest hill - solar ...",0.26,0,None,0.0,0.0,0.0,-1.0,-1.0,...,-0.068700,0.095760,-0.023533,-0.359177,0.123300,-0.619271,-0.111063,0.237274,-0.203963,0.137003
4545,mssrs r miller ltd,arkleby mill,1.00,0,None,0.0,0.0,0.0,-1.0,-1.0,...,0.143271,0.027659,-0.093508,-0.197086,0.211079,-0.324620,0.026372,-0.197345,-0.182225,0.017340
9321,edbro plc,"edbro, lever street - solar panels",1.70,0,None,0.0,0.0,0.0,-1.0,-1.0,...,0.200514,0.096243,-0.205161,-0.248463,0.338824,-0.364581,0.073330,0.053150,-0.084748,0.051679
7604,sands of luce holiday park,"sandmill, sandhead - solar panels",0.46,0,None,0.0,0.0,0.0,-1.0,-1.0,...,0.120344,-0.061660,-0.198754,-0.340019,0.308898,-0.360219,-0.162167,-0.235214,-0.275878,-0.005251


In [78]:
def save_to_csv(df: pd.DataFrame, name="df"):
    ts = datetime.now(ZoneInfo("Europe/London")).strftime("%H%M%S_%d%m%Y")
    fname = f"{name}_{ts}.csv"
    df.to_csv(fname, index=False)
    print("Saved:", fname)

In [79]:
A = X_train.to_numpy()

num_df = X_train.select_dtypes(include=[np.number])
A = num_df.to_numpy(dtype=float)

print("numeric cols:", num_df.shape[1])
print("dtype:", A.dtype)
print("min/max:", np.nanmin(A), np.nanmax(A))
print("std:", np.nanstd(A))
print("fraction exactly zero:", np.mean(A == 0))
print("nonzeros per row (first 10):", np.count_nonzero(A, axis=1)[:10])

numeric cols: 1201
dtype: float64
min/max: -4.782886028289795 13500000.0
std: 9939.768868190109
fraction exactly zero: 0.34889813908678113
nonzeros per row (first 10): [782 782 781 782 782 781 782 782 782 782]


Now to apply these steps to the test set (after encoding it).

Embedding the test dataset.

In [80]:
X_test = bert_encoding_tracker(columns_for_bert, X_test)

Epoch 1/1 done (TEXT_OPERATOR_AND_SITE_NAME) | elapsed: 00:50.1


In [81]:
print(f"New DataFrame Shape: {X_test.shape}")

New DataFrame Shape: (2055, 1206)


In [82]:
X_test.head()

,Operator (or Applicant),Site Name,Installed Capacity (MWelec),CHP Enabled,CfD Allocation Round,RO Banding (ROC/MWh),FiT Tariff (p/kWh),CfD Capacity (MW),Turbine Capacity,No. of Turbines,...,TEXT_OPERATOR_AND_SITE_NAME_emb_758,TEXT_OPERATOR_AND_SITE_NAME_emb_759,TEXT_OPERATOR_AND_SITE_NAME_emb_760,TEXT_OPERATOR_AND_SITE_NAME_emb_761,TEXT_OPERATOR_AND_SITE_NAME_emb_762,TEXT_OPERATOR_AND_SITE_NAME_emb_763,TEXT_OPERATOR_AND_SITE_NAME_emb_764,TEXT_OPERATOR_AND_SITE_NAME_emb_765,TEXT_OPERATOR_AND_SITE_NAME_emb_766,TEXT_OPERATOR_AND_SITE_NAME_emb_767
9526,liscombe park limited,liscombe business park - ground mounted solar ...,0.27,0,None,0.0,0.0,0.0,-1.0,-1.0,...,0.041704,-0.122957,-0.037965,-0.290829,0.164246,-0.416498,-0.165805,0.025041,-0.193168,-0.005341
2942,acorn energy b7 ltd,aghamore,15.00,0,None,0.0,0.0,0.0,3.0,5.0,...,0.003203,-0.099455,-0.027721,-0.220630,0.157190,-0.278555,0.148246,-0.192801,0.100761,0.097710
10959,cedar retreats,"cedar retreats, west tanfield - solar panels",0.17,0,None,0.0,0.0,0.0,-1.0,-1.0,...,-0.172184,-0.184857,-0.177150,-0.381181,0.159918,-0.330564,-0.069154,0.109305,-0.013057,0.060271
502,energy developments (uk) ltd/cleanaway,sandy lane generation plant,2.00,0,None,0.0,0.0,0.0,-1.0,-1.0,...,0.143018,0.045473,-0.168828,-0.282235,0.252277,-0.200705,0.114732,-0.069273,-0.183504,0.001208
2236,private developer,treworgans wind turbine,2.00,0,None,0.0,0.0,0.0,2.0,1.0,...,0.049524,-0.040413,-0.196261,-0.185155,0.057447,-0.381316,0.026938,-0.206932,-0.176941,-0.132800


## Normalisation of Only BERT-embedded Columns

Note: You need to convert the Pandas DataFrame object (the bert columns here) to an 2-d NumPy Array for normalize() to work. Also note, normalize() works row by row, so sample by sample, and not by columns (aka by features). So you must normalise all bert columns together, as below.

In [83]:
def normalize_bert_embeddings(df: pd.DataFrame):
    # Identify columns that were generated as embeddings.
    emb_columns = [col for col in df.columns if '_emb_' in col]
    
    # Normalise only the embedding columns (row-wise).
    df[emb_columns] = normalize(df[emb_columns].values, norm="l2", axis=1)
    
    # Check the result.
    print(df[emb_columns].head())
    return df
    
# Now normalise the embeddings.
X_train = normalize_bert_embeddings(X_train)

       TEXT_OPERATOR_AND_SITE_NAME_emb_0  TEXT_OPERATOR_AND_SITE_NAME_emb_1  \
2266                            0.003796                           0.011143   
11305                           0.027206                           0.025268   
4545                            0.006121                           0.042116   
9321                            0.059907                           0.012532   
7604                            0.018046                           0.041652   

       TEXT_OPERATOR_AND_SITE_NAME_emb_2  TEXT_OPERATOR_AND_SITE_NAME_emb_3  \
2266                            0.028860                          -0.053555   
11305                           0.050925                          -0.057779   
4545                            0.040365                          -0.059233   
9321                            0.048840                          -0.058761   
7604                            0.053456                          -0.058178   

       TEXT_OPERATOR_AND_SITE_NAME_emb_4  TEXT_OPE

In [84]:
X_test = normalize_bert_embeddings(X_test)

       TEXT_OPERATOR_AND_SITE_NAME_emb_0  TEXT_OPERATOR_AND_SITE_NAME_emb_1  \
9526                            0.029148                           0.006254   
2942                            0.013368                           0.009770   
10959                           0.003529                           0.024297   
502                             0.027941                          -0.008003   
2236                            0.025376                           0.018302   

       TEXT_OPERATOR_AND_SITE_NAME_emb_2  TEXT_OPERATOR_AND_SITE_NAME_emb_3  \
9526                            0.054259                          -0.035109   
2942                            0.017324                          -0.048353   
10959                           0.036560                          -0.012953   
502                             0.039801                          -0.050790   
2236                            0.040206                          -0.065474   

       TEXT_OPERATOR_AND_SITE_NAME_emb_4  TEXT_OPE

The same normalisation is then applied to the validation and test sets.

In [85]:
X_train[columns_for_bert].head(3)

,TEXT_OPERATOR_AND_SITE_NAME
2266,[Operator (or Applicant)] greencoat [Site Name...
11305,[Operator (or Applicant)] st johns beaumont sc...
4545,[Operator (or Applicant)] mssrs r miller ltd [...


In [86]:
X_test[columns_for_bert].head(3)

,TEXT_OPERATOR_AND_SITE_NAME
9526,[Operator (or Applicant)] liscombe park limite...
2942,[Operator (or Applicant)] acorn energy b7 ltd ...
10959,[Operator (or Applicant)] cedar retreats [Site...


## Dropping Columns Post-Embedding/Encoding

In [87]:
drop_cols = [
    "TEXT_OPERATOR_AND_SITE_NAME",
    "Operator (or Applicant)",
    "Site Name",
    "Region",
    "Technology Type", 
    "Storage Type",
    "Planning Authority",
    "Offshore Wind Round", # I have "Round_Rank" so I can drop this.
    "CfD Allocation Round", # I have "CfD_Round_Rank" so I can drop this.
    "Mounting Type for Solar" # I have "Round_Rank" so I can drop this.
]

# This catches anything we forgot to encode/drop.
for d in (X_train, X_test):
    d.drop(columns=[c for c in drop_cols if c in d.columns], inplace=True, errors="ignore")

Now the features are finalised, and cleaning and preprocessing has been completed, it is time to check the shape of our dataset. Remember we are roughly aiming for a 1:10 ratio in the number of features versus the number of instances, ideally, to avoid overfitting (important for our machine learning downstream).

In [88]:
X_train.shape[1]

1201

In [89]:
print(f"{round(X_train.shape[1]*100/X_train.shape[0], 2)} %")

14.62 %


The ratio is slightly above the rule-of-thumb threshold, but it is not a major concern.

## Final Checks

In [90]:
ml_ready_check(X_train, X_test)


=== X_train ===
shape: (8213, 1201)
non-numeric cols: 0
NaN count (numeric): 0
Inf count (numeric): 0
constant numeric cols: 0
duplicate rows: 1

=== X_test ===
shape: (2055, 1201)
non-numeric cols: 0
NaN count (numeric): 0
Inf count (numeric): 0
constant numeric cols: 62
  examples: ['Region_unknown', 'Technology Type_biomass (co-firing)', 'Technology Type_compressed air energy storage', 'Technology Type_flywheels', 'Technology Type_fuel cell (hydrogen)', 'Technology Type_hot dry rocks (hdr)', 'Technology Type_liquid air energy storage', 'Technology Type_tidal lagoon', 'Technology Type_unknown', 'Planning Authority_ards and north down', 'Planning Authority_argyll, bute and south lochaber', 'Planning Authority_bexley', 'Planning Authority_brighton and hove', 'Planning Authority_broxbourne', 'Planning Authority_camden']
duplicate rows: 0

=== Train/Test column alignment ===
same number of columns? True
missing in test: 0
extra in test: 0
same column order? True

✅ If there are no non-n

## Saving the Cleaned and Prepared Datasets

Saving the train dataset.

In [91]:
# Features
save_to_csv(X_train, "ENERGY embTrain Data post Normalisation FEATURES")

# Label
save_to_csv(y_train, "ENERGY embTrain Data post Normalisation LABEL")

Saved: ENERGY embTrain Data post Normalisation FEATURES_160813_07042026.csv
Saved: ENERGY embTrain Data post Normalisation LABEL_160819_07042026.csv


Saving the test dataset.

In [92]:
# Features
save_to_csv(X_test, "ENERGY embTest Data post Normalisation FEATURES")

# Label
save_to_csv(y_test, "ENERGY embTest Data post Normalisation LABEL")

Saved: ENERGY embTest Data post Normalisation FEATURES_160819_07042026.csv
Saved: ENERGY embTest Data post Normalisation LABEL_160821_07042026.csv


# Conclusion of Part 1: Data Readiness

In this notebook, we have successfully transformed a raw, complex dataset into a robust, machine-readable format ready for predictive modelling.

Key achievements in this phase include:

- Rigorous Data Cleaning: Systematically identifying and resolving structural anomalies (such as bizarre Excel date artifacts within the Wind Rounds) and standardising formats across the entire dataset.
- Leakage Prevention: Removing projects with 'Pending' or 'Withdrawn' statuses to ensure our model only learns from definitive 'Approved' or 'Refused' outcomes.
- Feature Engineering: Using NLP (BERT embeddings) to extract semantic meaning from unstructured text fields like site and operator names, capturing nuances that standard categorical encoding would miss.
- Robust Imputation & Scaling: Systematically handling missing values across different feature types (Zero-fill vs. Negative-One-fill vs. Medians) and applying standardisation to ensure mathematical stability for deep learning.

We have exported our final, clean feature matrix (X) and target labels (y). In Part 2, we will load this prepared data to tackle the core machine learning challenge: building a model capable of finding the hidden patterns of planning refusals within a severely imbalanced dataset.